# Pixel-scale (sub-cellular) per-cell features for Track B1 (`future_plan.md` §18.10)

Feature extraction only. The analysis runs in `multiscale-audit`.

**Inputs:** the images and masks from `danenberg2022` / `jackson2020`, plus the cell identities from the prep kernels. The cells are those of every other run: tumour images, Danenberg hot aggregates removed, ≥ 200 cells per image.

**Per cell, for the 14 shared channels** (11 markers + H3, DNA1, DNA2):
- `raw`: the mean raw intensity under the full mask (the usual "cell mean" truth and inputs).
- **Pixel features**, on arcsinh(pixel / 5) and computed separately under the full mask and under the mask eroded by 2 px (`e2`, the spillover control):
  - `dio`: mean of the outer third minus the inner third of the cell, by relative distance to the cell boundary (membrane vs centre);
  - `sd`: within-cell standard deviation;
  - `mi`: within-cell Moran's I over 4-connected pixel pairs (spatial texture).
- **Shape** (full mask): perimeter pixels, maximum inscribed distance, eccentricity.
- `band`: mean arcsinh signal in the 2-px band just outside the cell. This is the neighbour-ring spillover control: in `multiscale-audit`, pixel features must add beyond it.

A mask hash per image is saved, for comparison with `r5-validate`.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, hashlib, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy import ndimage as ndi
from joblib import Parallel, delayed
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs"], check=True)   # newer Python images
import tifffile, imagecodecs
warnings.filterwarnings("ignore")
print("python", sys.version.split()[0], "| numpy", np.__version__, "| imagecodecs", imagecodecs.__version__)

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261105
MIN_CELLS = 200
COF = 5.0
SMOKE_IMAGES = 20
NUC = ["histone_h3", "dna1", "dna2"]
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)
assert len(MARKERS) == 11
IN_SET = NUC + MARKERS
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = p
IMGP, MASKP = {"danenberg": {}, "jackson": {}}, {"danenberg": {}, "jackson": {}}
for p in g("*.tiff"):
    q = p.replace("\\", "/")
    name = os.path.splitext(os.path.basename(p))[0]
    co = "danenberg" if name.startswith("MB_") else "jackson"
    (MASKP if "/masks/" in q else IMGP)[co][name] = p
print(P, {c: (len(IMGP[c]), len(MASKP[c])) for c in IMGP})

def load_cells(key):
    cells = pd.read_parquet(P[key])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
    else:
        cells = cells[cells.is_tumour == 1]
    cells = cells[cells.image_name.isin(set(IMGP[key]) & set(MASKP[key]))]
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    if SMOKE:
        keep = sorted(keep)[::max(1, len(keep) // SMOKE_IMAGES)]
    cells = cells[cells.image_name.isin(keep)].sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    return cells, np.asarray(img_names), starts

TAB = {k: load_cells(k) for k in ("danenberg", "jackson")}
for k, (c, names, st) in TAB.items():
    print(f"{k}: {len(c):,} cells, {len(names)} images")

## Per-image pass

In [ ]:
def erode_once(mask):
    mn = ndi.minimum_filter(mask, size=3, mode="nearest")
    mx = ndi.maximum_filter(mask, size=3, mode="nearest")
    return np.where(mn == mx, mask, 0)

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def cell_stats(mk, lut, A, n, W_):
    """dio / sd / mi per channel and shape features for label image mk (cells indexed by lut)."""
    flat_lab = mk.ravel()
    L = lut[flat_lab]
    sel = np.flatnonzero(L >= 0)
    li = L[sel]
    V = A.reshape(-1, A.shape[-1])
    cnt = np.bincount(li, minlength=n).astype(np.float64)
    ok = cnt > 0
    def cmean(x, idx, lab):
        c = np.bincount(lab, minlength=n)
        return np.where(c > 0, np.bincount(lab, x[idx], n) / np.maximum(c, 1), np.nan)
    interior = (ndi.minimum_filter(mk, 3, mode="nearest") == ndi.maximum_filter(mk, 3, mode="nearest")) & (mk > 0)
    dist = ndi.distance_transform_edt(interior).ravel()[sel]          # 0 on boundary pixels
    dmax = np.zeros(n)
    np.maximum.at(dmax, li, dist)
    rel = dist / np.maximum(dmax[li], 1.0)
    outer, inner = rel < 1 / 3, rel >= 2 / 3
    # 4-connected pixel pairs inside the same cell
    H_ = mk.shape[0]
    idx = np.arange(mk.size)
    hp = np.flatnonzero((L[:-1] >= 0) & (L[:-1] == L[1:]) & ((idx[:-1] % W_) != W_ - 1))
    vp = np.flatnonzero((L[:-W_] >= 0) & (L[:-W_] == L[W_:]))
    pa = np.r_[hp, vp]
    pb = np.r_[hp + 1, vp + W_]
    pl = L[pa]
    npair = np.bincount(pl, minlength=n).astype(np.float64)
    out = {}
    for j in range(V.shape[1]):
        x = V[:, j].astype(np.float64)
        m = np.where(ok, np.bincount(li, x[sel], n) / np.maximum(cnt, 1), np.nan)
        dx = x[sel] - m[li]
        var = np.bincount(li, dx * dx, n) / np.maximum(cnt, 1)
        out[f"sd__{j}"] = np.where(ok, np.sqrt(var), np.nan)
        out[f"dio__{j}"] = cmean(x, sel[outer], li[outer]) - cmean(x, sel[inner], li[inner])
        xa, xb = x[pa] - m[pl], x[pb] - m[pl]
        num = np.bincount(pl, xa * xb, n)
        den = np.bincount(li, dx * dx, n)
        with np.errstate(invalid="ignore", divide="ignore"):
            out[f"mi__{j}"] = np.where((npair > 0) & (den > 0), (cnt / np.maximum(npair, 1)) * num / den, np.nan)
    # shape
    yy, xx = np.divmod(sel, W_)
    my, mx_ = np.bincount(li, yy, n) / np.maximum(cnt, 1), np.bincount(li, xx, n) / np.maximum(cnt, 1)
    cyy = np.bincount(li, (yy - my[li]) ** 2, n) / np.maximum(cnt, 1)
    cxx = np.bincount(li, (xx - mx_[li]) ** 2, n) / np.maximum(cnt, 1)
    cxy = np.bincount(li, (yy - my[li]) * (xx - mx_[li]), n) / np.maximum(cnt, 1)
    tr, det = cyy + cxx, cyy * cxx - cxy ** 2
    disc = np.sqrt(np.maximum(tr ** 2 / 4 - det, 0))
    l1, l2 = tr / 2 + disc, np.maximum(tr / 2 - disc, 0)
    out["ecc"] = np.where(ok & (l1 > 0), np.sqrt(1 - l2 / np.maximum(l1, 1e-9)), np.nan)
    out["perim"] = np.bincount(li, (~interior.ravel()[sel]).astype(np.float64), n)
    out["dmax"] = dmax
    out["npix"] = cnt
    return out

def band_means(mask, lut, A, n):
    """Mean arcsinh signal over the 2-px band outside each cell (pixels not in the cell, within 2 px of it)."""
    BIG = np.iinfo(np.int64).max
    mx = ndi.maximum_filter(mask, size=5, mode="constant", cval=0)
    mn = ndi.minimum_filter(np.where(mask > 0, mask, BIG), size=5, mode="constant", cval=BIG)
    V = A.reshape(-1, A.shape[-1]).astype(np.float64)
    m_f, mx_f, mn_f = mask.ravel(), mx.ravel(), mn.ravel()
    labs, pix = [], []
    for cand in (mx_f, mn_f):
        c = np.where(cand == BIG, 0, cand)
        hit = np.flatnonzero((c > 0) & (c != m_f))
        Lc = lut[np.minimum(c[hit], len(lut) - 1)]
        Lc = np.where(c[hit] < len(lut), Lc, -1)
        keep = Lc >= 0
        labs.append(Lc[keep]); pix.append(hit[keep])
    lab, px = np.concatenate(labs), np.concatenate(pix)
    # a pixel counted twice for the same cell (when max == min) is de-duplicated
    key = px.astype(np.int64) * (n + 1) + lab
    _, first = np.unique(key, return_index=True)
    lab, px = lab[first], px[first]
    c = np.bincount(lab, minlength=n)
    return {f"band__{j}": np.where(c > 0, np.bincount(lab, V[px, j], n) / np.maximum(c, 1), np.nan) for j in range(V.shape[1])}

def image_pass(img_path, mask_path, obj, n_chan, chans):
    mask_raw = tifffile.imread(mask_path)
    h = hashlib.md5(np.ascontiguousarray(mask_raw).tobytes()).hexdigest()
    mask = mask_raw.astype(np.int64)
    im = read_hwc(img_path, n_chan)[..., chans]
    assert im.shape[:2] == mask.shape, (img_path, im.shape, mask.shape)
    n = len(obj)
    lut = np.full(max(int(mask.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(n)
    W_ = mask.shape[1]
    # raw means (full mask)
    L = lut[mask.ravel()]
    sel = np.flatnonzero(L >= 0)
    cnt = np.bincount(L[sel], minlength=n)
    flat = im.reshape(-1, im.shape[-1])
    raw = np.stack([np.bincount(L[sel], flat[sel, j], n) for j in range(flat.shape[1])], 1) / np.maximum(cnt, 1)[:, None]
    raw[cnt == 0] = np.nan
    A = np.arcsinh(im / COF)
    res = {f"raw__{j}": raw[:, j] for j in range(raw.shape[1])}
    full = cell_stats(mask, lut, A, n, W_)
    res.update({f"{k}" if k in ("ecc", "perim", "dmax", "npix") else f"{k.split('__')[0]}_full__{k.split('__')[1]}": v
                for k, v in full.items()})
    e2 = cell_stats(erode_once(erode_once(mask)), lut, A, n, W_)
    res.update({f"{k.split('__')[0]}_e2__{k.split('__')[1]}": v for k, v in e2.items() if "__" in k})
    res["npix_e2"] = e2["npix"]
    res.update(band_means(mask, lut, A, n))
    return {k: np.asarray(v, np.float32) for k, v in res.items()}, h

for k, (cells, names, st) in TAB.items():
    chan = D_CHAN if k == "danenberg" else J_CHAN
    chans = [chan.index(c) for c in IN_SET]
    obj = cells.object_number.values.astype(np.int64)
    out = Parallel(n_jobs=os.cpu_count())(delayed(image_pass)(IMGP[k][s], MASKP[k][s], obj[a:b], len(chan), chans)
                                          for s, a, b in zip(names, st[:-1], st[1:]))
    cols = {c: np.concatenate([o[0][c] for o in out]) for c in out[0][0]}
    hashes = [o[1] for o in out]
    del out
    gc.collect()
    rename = {}
    for c in list(cols):
        if "__" in c:
            base, j = c.split("__")
            rename[c] = f"{base}__{IN_SET[int(j)]}"
    df = pd.DataFrame({rename.get(c, c): v for c, v in cols.items()})
    df.insert(0, "object_number", obj)
    df.insert(0, "image_name", cells.image_name.values)
    print(f"{k}: cells with ≥1 pixel {np.mean(df.npix > 0):.4f}, with ≥1 eroded pixel {np.mean(df.npix_e2 > 0):.4f}, "
          f"median pixels {df.npix.median():.0f} / {df.npix_e2.median():.0f}")
    assert np.mean(df.npix > 0) > 0.95, "mask relabelling does not match the cell table"
    chk = []
    for c in ("histone_h3", "dna1", "pan_cytokeratin", "cd45"):
        tc = D_TABLE.get(c, c) if k == "danenberg" else c
        a, b = df[f"raw__{c}"].values.astype(np.float64), cells[tc].values.astype(np.float64)
        ok = np.isfinite(a) & np.isfinite(b)
        chk.append(dict(cohort=k, channel=c, r=float(np.corrcoef(a[ok], b[ok])[0, 1])))
    chk = pd.DataFrame(chk)
    print(chk.round(3).to_string(index=False))
    assert chk.r.median() > 0.6, "image-derived cell means disagree with the published table"
    nan_frac = df.drop(columns=["image_name", "object_number"]).isna().mean().sort_values(ascending=False)
    print("highest NaN fractions:", nan_frac.head(6).round(3).to_dict())
    df.to_parquet(f"{OUT}/pixfeat_{k}.parquet", index=False)
    pd.DataFrame(dict(image_name=names, mask_md5=hashes)).to_csv(f"{OUT}/mask_hashes_{k}.csv", index=False)
    tick(f"{k}: {df.shape[1] - 2} features saved")
json.dump({"smoke": SMOKE, "minutes": (time.time() - T0) / 60, "cof": COF, "in_set": IN_SET}, open(f"{OUT}/run_info.json", "w"))